# RUIP_BA_DrawingData: Valid claim and Invlid claim 
This program is primarily configured and tuned for the drawing dataset (applying RP+Laplace noise), and most model parameters and hyperparameters are optimized for this setting. When applying the same framework to other datasets, these parameters generally need to be adjusted and re-tuned to ensure optimal performance.

The program processes oversampled training data from Group 1 by first projecting it into a lower-dimensional space using random projection. To ensure differential privacy, either Laplace or Gaussian noise is added to the projected features. The resulting privacy-preserved data is then used to train a machine learning model.

For the valid claim setting, the trained model is evaluated using test data from Group 1. The test data is transformed using the same random projection matrices and the same differential privacy configuration as used during training, ensuring full consistency between training and testing procedures.

For the invalid claim setting, the trained model is evaluated using test data from Group 2. In this case, a different random projection matrix is used for Group 2, introducing inconsistency in the feature transformation relative to the training pipeline. However, the differential privacy parameters (e.g., $\varepsilon$, $\delta$, and noise scale) remain unchanged, as they are treated as fixed and publicly known across both groups.

Overall, this program supports both valid and invalid claim settings across different datasets. It can operate on plain data as well as in configurations involving random projection and/or differential privacy noise (Laplace or Gaussian), with appropriate adjustment of model parameters and hyperparameters.

In [1]:
# =====================================================
# Hyperparameters and Configuration for model training
# =====================================================

# Dataset paths
DATASET_PATH = 'Dataset/OversampledDrawingData.csv'
TESTDATASET_PATH = 'Dataset/DrawingDatatest.csv'

# Profile separation
SEPARATE_PROFILE = 155

# Random projection parameters
USE_RANDOM_PROJECTION = True

if USE_RANDOM_PROJECTION==False:
    N_COMPONENTS = 65
else:
   N_COMPONENTS = 46
   

# Differential privacy parameters
USE_LAPLACE_NOISE = True
USE_GAUSSIAN_NOISE = False
EPSILON = 7.0
SENSITIVITY = 1.0
DELTA = 1e-5

# Input dimension for neural network
INPUT_DIM = N_COMPONENTS
BATCH_SIZE = 32
EPOCHS = 50

# Dynamic column generation
column1 = [f'RPF{i+1}' for i in range(N_COMPONENTS)]

# =====================================================
# Load dataset
# =====================================================

import pandas as pd
import numpy as np

dataset = pd.read_csv(DATASET_PATH, index_col=0)

In [2]:
# Separate the profiles into two groups:
# (i) training profiles (0-SEPARATE_PROFILE-1)
# (ii) auxiliary profiles (SEPARATE_PROFILE and above)

totalUser = len(pd.unique(dataset['Label']))

trainingData = dataset[dataset['Label'] < SEPARATE_PROFILE]
auxilaryData = dataset[dataset['Label'] >= SEPARATE_PROFILE]

print("Total user in training dataset:", len(pd.unique(trainingData['Label'])))
print("Total user in auxiliary dataset:", len(pd.unique(auxilaryData['Label'])))

Total user in training dataset: 155
Total user in auxiliary dataset: 38


In [3]:
#Random Project
from sklearn.random_projection import SparseRandomProjection

def apply_random_projection(data, total_user, n_components=N_COMPONENTS):
    
    datasetRP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):  
        rng = np.random.RandomState(seed)
        X = data[data['Label']==seed]
        transformer = SparseRandomProjection(n_components=N_COMPONENTS, random_state=rng)
        Xdata=X.drop(columns=['Label'])
        XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column1)
        XRP['Label']=seed
        datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)    
    return datasetRP

In [4]:
# Laplace noise
def add_laplace_noise(data, total_user, epsilon=EPSILON, sensitivity=SENSITIVITY):
    datasetRPDP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        X = data[data['Label']==seed]
        Xdata=X.drop(columns=['Label'])
        
        scale = sensitivity / epsilon
        noise = np.random.laplace(loc=0.0, scale=scale, size=Xdata.shape)
        Xdata=Xdata+noise 
        
        XdataRPDP = pd.DataFrame(Xdata,columns=column1)
        XdataRPDP['Label']=seed
        datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)
    return datasetRPDP

In [5]:
# Gaussioan noise
def add_gaussian_noise(data, total_user,epsilon=EPSILON,sensitivity=SENSITIVITY,delta=DELTA):
    datasetRPDP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user):
        X = data[data['Label']==seed]
        Xdata=X.drop(columns=['Label'])
        
        sigma = (sensitivity * np.sqrt(2 * np.log(1.25 / delta))) / epsilon
        noise = np.random.normal(0, sigma)
        Xdata=Xdata+noise 
        
        XdataRPDP = pd.DataFrame(Xdata,columns=column1)
        XdataRPDP['Label']=seed
        datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)
    return datasetRPDP

In [6]:
# =====================================================
# Feature Processing Pipeline
# =====================================================

feature_data = trainingData
total_user=len(pd.unique(trainingData['Label']))
# Apply Random Projection
if USE_RANDOM_PROJECTION:
    feature_data = apply_random_projection(feature_data, total_user)

# Apply Laplace Noise
if USE_LAPLACE_NOISE:
    feature_data = add_laplace_noise(feature_data, total_user)

# Apply Gaussian Noise
if USE_GAUSSIAN_NOISE:
    feature_data = add_gaussian_noise(feature_data, total_user)

# Convert processed features to DataFrame
feature_data = pd.DataFrame(feature_data)


# Final processed dataset
processed_dataset = feature_data

#print("Total user in the training dataset:", len(pd.unique(processed_dataset['Label'])))
#print(processed_dataset.head())

C:\Users\mdmor\AppData\Local\Temp\ipykernel_46580\1979874663.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)
C:\Users\mdmor\AppData\Local\Temp\ipykernel_46580\1620265370.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)


In [7]:
#Prepare the traning data for training and testing the model
import tensorflow
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

if USE_RANDOM_PROJECTION==False:
    X=trainingData.drop(columns=['Label'])
    y=trainingData['Label']
else:
    X=processed_dataset.drop(columns=['Label'])
    y=processed_dataset['Label']

Xtrain, Xval, ytrain, yval = train_test_split(X, y, test_size=0.2, random_state=22)

ytrain = to_categorical(ytrain)
yval = to_categorical(yval)
print(Xtrain.shape)
print(ytrain.shape)
print(Xval.shape)
print(yval.shape)

(37232, 46)
(37232, 155)
(9308, 46)
(9308, 155)


In [8]:
# import all necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#matplotlib inlineimport keras
from keras.layers import Dense, Dropout, Input,Activation,Dropout, Flatten
from keras.models import Model,Sequential
from keras.datasets import mnist

from keras.layers import BatchNormalization
from keras.optimizers import SGD, RMSprop, Adam
def adam_optimizer():
    return Adam(learning_rate=0.0002, beta_1=0.5)

def RMSprop_optimizer():
    return RMSprop(learning_rate=0.001, rho=0.9)

In [9]:
#neural network architecture

def create_classifierRP(release=False, totalClass=SEPARATE_PROFILE):
  classifier = Sequential()
  classifier.add(Dense(64, input_dim=N_COMPONENTS))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.3))

  classifier.add(Dense(128))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.2))

  classifier.add(Dense(64))
  classifier.add(BatchNormalization())
  classifier.add(Activation('relu'))
  classifier.add(Dropout(0.3))


  classifier.add(Dense(totalClass, activation='softmax'))

  classifier.compile(loss='categorical_crossentropy', optimizer=RMSprop_optimizer(),metrics=['accuracy'])
  return classifier

Clasf=create_classifierRP()
Clasf.summary()

C:\Users\mdmor\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         3,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 155)            │        10,075 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,683 (119.86 KB)

 Trainable params: 30,171 (117.86 KB)

 Non-trainable params: 512 (2.00 KB)

In [10]:
#Train the classifier
from keras.callbacks import ReduceLROnPlateau

learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', patience=5, verbose=1, factor=0.5,min_lr=0.0001)
callbacks_list = [learning_rate_reduction]

Classfier2= create_classifierRP(True,155)

#------Comment will start from here
lossc='categorical_crossentropy'
optimizerc=RMSprop(learning_rate=0.001, rho=0.9)
Classfier2.compile(loss=lossc, optimizer=optimizerc,metrics=['accuracy'])
#------Comments will end from here
historyc2 =  Classfier2.fit(Xtrain, ytrain, batch_size=64, epochs=200, validation_data=(Xval, yval),verbose=1)

Epoch 1/200
582/582 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.2486 - loss: 3.8688 - val_accuracy: 0.9741 - val_loss: 0.5821
Epoch 2/200
582/582 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7430 - loss: 1.1198 - val_accuracy: 0.9908 - val_loss: 0.0971
Epoch 3/200
582/582 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8139 - loss: 0.6753 - val_accuracy: 0.9940 - val_loss: 0.0466
Epoch 4/200
582/582 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8477 - loss: 0.5328 - val_accuracy: 0.9946 - val_loss: 0.0346
Epoch 5/200
582/582 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8671 - loss: 0.4601 - val_accuracy: 0.9953 - val_loss: 0.0252
Epoch 6/200
582/582 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8760 - loss: 0.4322 - val_accuracy: 0.9957 - val_loss: 0.0224
Epoch 7/200
582/582 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8810 - loss: 0.4028 - val_accuracy: 0.9965 - val_loss: 0.0202
Epoch 8/200
582/582 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8886 - loss: 0.3755 - val_accu

In [27]:
# =====================================================
# Hyperparameters and Configuration for valid and invalid claim
# =====================================================
#Type of claim
VALID_CLAIM = True
INVALID_CLAIM =False

In [28]:
#Random Project for vlaid and invalid claim
from sklearn.random_projection import SparseRandomProjection

def apply_random_projectionClaim(data, total_user, n_components=N_COMPONENTS):
    
    datasetRP = pd.DataFrame(columns=column1)

    for seed in range(0,total_user): 
        if VALID_CLAIM==True: 
            rng = np.random.RandomState(seed)
        else:
             rng = np.random.RandomState(seed+SEPARATE_PROFILE)
        X = data[data['Label']==seed]
        transformer = SparseRandomProjection(n_components=N_COMPONENTS, random_state=rng)
        Xdata=X.drop(columns=['Label'])
        XRP = pd.DataFrame(transformer.fit_transform(Xdata),columns=column1)
        XRP['Label']=seed
        datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)    
    return datasetRP

In [29]:
#read the test data either for valid or invalid claim
import csv
import pandas as pd

if VALID_CLAIM:
    testdataset = pd.read_csv(TESTDATASET_PATH, index_col=0)
    testdataset = testdataset[testdataset['Label'] < SEPARATE_PROFILE]
    
if INVALID_CLAIM:
    testdataset = pd.read_csv(TESTDATASET_PATH, index_col=0)
    testdataset = testdataset[testdataset['Label'] >= SEPARATE_PROFILE]
    newID = np.random.randint(0, SEPARATE_PROFILE, size=testdataset.shape[0])
    testdataset['Label'] = newID
    
testdataset.head()

,Label,1,2,3,4,5,6,7,8,9,...,56,57,58,59,60,61,62,63,64,65
3799,27,0.024390,0.095833,0.259740,0.157025,0.209877,0.135246,0.234568,0.093117,0.132530,...,0.951830,0.950914,0.950914,0.949087,0.927700,0.128049,0.272,0.0,0.046512,0.076503
18582,138,0.008130,0.054167,0.090909,0.111570,0.061728,0.118852,0.119342,0.028340,0.016064,...,0.618497,0.617902,0.617902,0.616715,0.602817,0.082317,0.180,0.0,0.534884,0.010929
6423,47,0.073171,0.333333,0.229437,0.161157,0.131687,0.413934,0.604938,0.267206,0.188755,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.152439,0.672,1.0,0.395349,0.398907
6892,51,0.077236,0.254167,0.056277,0.033058,0.032922,0.106557,0.016461,0.072874,0.148594,...,0.383430,0.383061,0.382098,0.379443,0.366197,0.054878,0.184,0.0,0.279070,0.038251
2885,21,0.105691,0.304167,0.346320,0.458678,0.452675,0.262295,0.106996,0.129555,0.120482,...,0.222543,0.218479,0.218479,0.218060,0.212207,0.564024,0.096,1.0,0.116279,0.098361


In [30]:
# =====================================================
# Feature Processing Pipeline
# =====================================================

feature_data = testdataset
total_user=len(pd.unique(testdataset['Label']))


# Apply Random Projection
if USE_RANDOM_PROJECTION:
    feature_data = apply_random_projectionClaim(feature_data, total_user)

# Apply Laplace Noise
if USE_LAPLACE_NOISE:
    feature_data = add_laplace_noise(feature_data, total_user)

# Apply Gaussian Noise
if USE_GAUSSIAN_NOISE:
    feature_data = add_gaussian_noise(feature_data, total_user)

# Convert processed features to DataFrame
feature_data = pd.DataFrame(feature_data)

# Final processed dataset
processed_dataset = feature_data

#print("Total user in the training dataset:", len(pd.unique(processed_dataset['Label'])))
#print(processed_dataset.head())

C:\Users\mdmor\AppData\Local\Temp\ipykernel_46580\432162272.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRP = pd.concat([datasetRP, XRP], ignore_index=True)
C:\Users\mdmor\AppData\Local\Temp\ipykernel_46580\1620265370.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  datasetRPDP = pd.concat([datasetRPDP, XdataRPDP], ignore_index=True)


In [31]:
if USE_RANDOM_PROJECTION==False:
    Xtest=testdataset.drop(columns=['Label'])
    ytest=testdataset['Label']
else:
    Xtest=processed_dataset.drop(columns=['Label'])
    ytest=processed_dataset['Label']
    
ytest = to_categorical(ytest)

In [32]:
#Performance of the classifier
Classfier2.compile(loss='categorical_crossentropy', optimizer=Adam(),metrics=['accuracy'])
loss, accuracy = Classfier2.evaluate(Xtest, ytest)
print('Loss:', loss)
print('Accuracy:', accuracy)

129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9975 - loss: 0.0064
Loss: 0.004530458245426416
Accuracy: 0.9987866878509521
